In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

pd.set_option("display.max_columns", None, "display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 100
from src import config as C

# 03 — Veri Temizleme (Data Cleaning)

**Altın kural:** ham veriyi değiştirmiyoruz → `df = df_raw.copy()`. Temizleme mantığının tamamı `src/data_cleaning.py` içinde; bu notebook her adımı **gösteriyor ve neden yaptığını açıklıyor**, sonunda modülün sonucunu referans temiz veriyle doğruluyor.

> **Neden fonksiyon/modül?** Aynı temizliği hem burada, hem testlerde, hem yeni gelen veriyle tekrarlayabilmek için. Notebook'ta elle yazılan temizlik tekrar üretilemez.

In [2]:
from src.data_cleaning import read_raw, clean_data
raw = read_raw()
df = raw.copy()   # Ham veriyi korumak için kopya. Bundan sonra df üzerinde çalışırız.
print("Ham:", raw.shape)

Ham: (1820, 16)


## Adım 1 — Ücret sütunu: metin → sayı (`pd.to_numeric(errors="coerce")`)
`errors="coerce"` sayıya çevrilemeyen değeri hata vermek yerine `NaN` yapar. Ama dikkat: **kör `coerce` bilgi kaybettirir.** Önce ne kaybedeceğimize bakalım:

In [3]:
naive = pd.to_numeric(df["monthly_fee_tl"], errors="coerce")
lost = df.loc[naive.isna() & df["monthly_fee_tl"].notna(), ["customer_id", "plan", "discount_rate", "monthly_fee_tl"]]
display(lost)

,customer_id,plan,discount_rate,monthly_fee_tl
24,CUST100024,Premium,0.200,399TL
322,CUST100322,Premium,0.150,free
770,CUST100770,Standard,0.000,error
956,CUST100956,Standard,0.100,?
1492,CUST101492,Standard,0.100,—
1530,CUST101530,Basic,0.100,unknown
1673,CUST101673,Basic,0.100,four hundred


`399TL` sayıya çevrilebilir görünüyor (birim eki). Peki 399 doğru mu? Kurala göre bakalım: bu müşteri **Premium + %20 indirim** → doğru ücret `599 × 0.80 = 479.2`. Yani `399TL` bir birim yazım hatası değil, **tamamen yanlış değer**.

In [4]:
row = df[df.monthly_fee_tl == "399TL"][["customer_id", "plan", "discount_rate", "monthly_fee_tl"]]
row["kuralın_beklediği_ücret"] = row["plan"].map(C.PLAN_BASE_PRICE) * (1 - row["discount_rate"])
display(row)

,customer_id,plan,discount_rate,monthly_fee_tl,kuralın_beklediği_ücret
24,CUST100024,Premium,0.200,399TL,479.200


> **Ders:** "Düzeltilebilir görünen" bir değeri, doğrulayabildiğin bir kuralla çapraz kontrol etmeden güvenme. Temizleme modülümüz bu yüzden: (1) metni sayıya çevirir, (2) aralık dışını eler, (3) plan×indirim kuralıyla **tutarlılık kontrolü** yapar ve sapanları kuralla değiştirir (hepsi loglanır).

## Adım 2 — Kategorileri standartlaştırma
`strip()` baştaki/sondaki boşluğu, `lower()` harf farkını giderir; sonra sözlükle kanonik yazıma çeviririz. **Sözlükte olmayan değer sessizce NaN olmasın diye** modül bunu loga yazar.

In [5]:
before = {c: raw[c].nunique() for c in C.CATEGORICAL_COLS}
res = clean_data(raw)
after = {c: res.clean[c].nunique() for c in C.CATEGORICAL_COLS}
display(pd.DataFrame({"önce_farklı_değer": before, "sonra_farklı_değer": after}))

,önce_farklı_değer,sonra_farklı_değer
region,7,6
device_type,3,3
plan,4,3
contract_type,4,3
payment_method,5,4


## Adım 3 — Aralık dışı değerler → `NaN`
Yaş = 130 gibi imkânsız değerleri **silmek** yerine *eksik* yaparız (`NaN`). Neden?
* Satırı silmek, o müşterinin diğer 14 doğru bilgisini de çöpe atar.
* Aralık dışı değeri medyanla **burada** doldurmak ise *veri sızıntısı* olur (aşağıda açıklıyoruz). Doldurmayı Pipeline'a bırakıyoruz.

## Adım 4 — Plan↔ücret çelişkisi ve ücretin geri kazanımı
Notebook 02'de ücret = plan fiyatı × (1 − indirim) kuralını keşfettik. Uygulama sırası önemli:
1. Geçerli ücret + indirim, başka bir planın liste fiyatına *kuruşu kuruşuna* denk geliyorsa **plan etiketi yanlıştır** (3 olası değerli etiketin hatalı olma ihtimali, rastgele bir ücretin başka bir planla tam eşleşme ihtimalinden çok daha yüksek).
2. Sonra ücret, düzeltilmiş plan ve indirimden **hesaplanır**; eksik/bozuk/sapan ücretler bununla değiştirilir.

In [6]:
display(res.log)

,adim,sutun,etkilenen,aciklama
0,baslangic,-,1820,ham satır sayısı
1,kategori_standardizasyonu,region,7,büyük/küçük harf-boşluk düzeltildi
2,kategori_standardizasyonu,plan,8,büyük/küçük harf-boşluk düzeltildi
3,kategori_standardizasyonu,contract_type,7,büyük/küçük harf-boşluk düzeltildi
4,kategori_standardizasyonu,payment_method,7,büyük/küçük harf-boşluk düzeltildi
5,aralik_disi_deger,age,6,"geçerli aralık [18, 100] dışı -> NaN"
6,aralik_disi_deger,monthly_usage_hours,4,"geçerli aralık [0, 100] dışı -> NaN"
7,aralik_disi_deger,satisfaction_score,2,"geçerli aralık [1, 10] dışı -> NaN"
8,ucret_birim_temizleme,monthly_fee_tl,1,'399TL' -> 399.0
9,ucret_metin_hatasi,monthly_fee_tl,6,"sayıya çevrilemeyen metin -> NaN: ?, error, fo..."


## Adım 5 — Tekrarlar
1. **Birebir tekrar** (12 satır): ilk kopyayı tut.
2. **Çelişkili ID** (8 müşteri, 16 satır): hangisinin doğru olduğunu zaman damgası olmadan bilemeyiz. İki seçenek var:

| Seçenek | Sonuç | Sorun |
|---|---|---|
| Çelişkili ID'lerin **hepsini sil** | 1820 − 12 − 16 = **1792** satır | 8 gerçek müşteriyi tamamen kaybederiz |
| **İlk kaydı tut** (dosya sırasına göre) | **1800** satır | Bir *varsayım* yapmış oluruz |

Varsayımımız: dosya sonuna eklenen ikinci kayıt geç gelen/yinelenen satırdır. Tüm çelişkiler `conflicting_customers.csv`'ye yazılır ki sonradan kontrol edilebilsin.

In [7]:
display(res.conflicts[["customer_id", "source_row", "monthly_usage_hours", "support_calls_6m", "kept"]].head(8))

,customer_id,source_row,monthly_usage_hours,support_calls_6m,kept
201,CUST100201,201,54.900,1,True
1818,CUST100201,1818,62.900,2,False
312,CUST100312,312,51.100,0,True
1815,CUST100312,1815,60.100,1,False
322,CUST100322,322,45.000,0,True
1814,CUST100322,1814,55.000,1,False
826,CUST100826,826,47.000,0,True
1813,CUST100826,1813,57.000,1,False


> ⚠️ **Dürüst not:** Bu bir varsayımdır. Aşağıda "referans temiz veri" ile karşılaştırınca ilk kaydın doğru olduğunu göreceğiz; **gerçek bir projede referansın olmaz** — CRM'den güncelleme zamanını almalı ya da iş birimine sormalısın.

## Adım 6 — Doğrulama: referans temiz veriyle karşılaştırma
Elimizde (ödev/sentetik veri olduğu için) gerçek değerleri içeren bir `clean_reference` dosyası var. Bunu **model eğitiminde ASLA kullanmıyoruz**; sadece temizleme kodumuzun doğruluğunu ölçmek için kullanıyoruz.

In [8]:
ref = pd.read_csv(C.REFERENCE_PATH).set_index("customer_id")
c = res.clean.set_index("customer_id").loc[ref.index]
rows = []
for col in ref.columns:
    ok = c[col].notna()
    if ref[col].dtype.kind in "if":
        diff = (~np.isclose(c.loc[ok, col].astype(float), ref.loc[ok.index[ok], col].astype(float))).sum()
    else:
        diff = (c.loc[ok, col].astype(str) != ref.loc[ok.index[ok], col].astype(str)).sum()
    rows.append({"sütun": col, "karşılaştırılan": int(ok.sum()), "referanstan_farklı": int(diff), "hâlâ_eksik(NaN)": int((~ok).sum())})
report = pd.DataFrame(rows); display(report)

,sütun,karşılaştırılan,referanstan_farklı,hâlâ_eksik(NaN)
0,age,1750,0,50
1,region,1768,6,32
2,device_type,1800,0,0
3,plan,1800,0,0
4,contract_type,1800,5,0
5,tenure_months,1800,0,0
6,monthly_usage_hours,1733,0,67
7,monthly_fee_tl,1800,0,0
8,discount_rate,1800,0,0
9,payment_method,1773,4,27


**Okuma:**
* `plan`, `monthly_fee_tl`, `age`, `monthly_usage_hours`, `satisfaction_score`, `churn` vb. → **0 fark**.
* `region`, `contract_type`, `payment_method` sütunlarında küçük harfle yazılmış hücrelerin (ör. `marmara`, `monthly`, `credit card`) bir kısmı referanstan farklı: bu hücrelerin **değeri de yanlışmış** (sadece yazımı değil). Bunu hiçbir kural yakalayamaz çünkü `marmara` geçerli bir bölge adıdır. Toplam etkilenen hücre sayısı çok küçük (aşağıda).

In [9]:
n_diff = int(report.loc[report.sütun.isin(["region", "contract_type", "payment_method"]), "referanstan_farklı"].sum())
total_cells = res.clean.shape[0] * (res.clean.shape[1] - 1)
display(Markdown(f"Referanstan farklı kalan hücre: **{n_diff}** / {total_cells:,} (**%{n_diff/total_cells:.3%}**). Bu, veri kalitesinde 'bilinmeyen bilinmeyen' sınıfıdır: tespit edemediğin hata."))

Referanstan farklı kalan hücre: **15** / 27,000 (**%0.056%**). Bu, veri kalitesinde 'bilinmeyen bilinmeyen' sınıfıdır: tespit edemediğin hata.

## Adım 7 — Eksik değerler: doldurmalı mıyız?
Burada **doldurmuyoruz.** Çünkü medyan/mod gibi değerleri *tüm veriden* hesaplayıp sonra train/test ayırırsak, test kümesinin istatistiği eğitime sızar. Buna **Data Leakage** denir. Doldurma, eğitim verisinden öğrenilen değerlerle **Pipeline içinde** yapılacak (Notebook 07).

Kalan eksikler:

In [10]:
miss = res.clean.isna().sum(); display(miss[miss > 0].to_frame("eksik_adet"))

,eksik_adet
age,50
region,32
monthly_usage_hours,67
payment_method,27
satisfaction_score,82


## Adım 8 — Kaydet

In [11]:
res.clean.to_csv(C.CLEAN_PATH, index=False)
res.conflicts.to_csv(C.CONFLICTS_PATH, index=False)
res.log.to_csv(C.CLEANING_LOG_PATH, index=False)
print("Kaydedildi:", C.CLEAN_PATH.relative_to(C.ROOT), res.clean.shape)

Kaydedildi: data/processed/customer_churn_clean.csv (1800, 16)
